# Verifier Bottleneck Experiments on Colab

Runtime: choose a GPU runtime before running. This notebook clones the experiment branch, installs dependencies, runs the selected experiments, and saves artifacts to Google Drive.

Set `MODE = "smoke"` for a quick end-to-end test. Set `MODE = "full"` for the real run.

In [ ]:
REPO_URL = "https://github.com/nZiben/verifier-bottleneck-math.git"
BRANCH = "experiment/game24-composition"

MODE = "full"  # "smoke" or "full"
RUN_GAME24 = True
RUN_MODP = True

DRIVE_DIR = "/content/drive/MyDrive/verifier-bottleneck-math-runs"
CLONE_DIR = "/content/verifier-bottleneck-math"

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys
from datetime import datetime

try:
    import torch
    print("GPU available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch precheck failed:", exc)

from google.colab import drive
drive.mount("/content/drive")
pathlib.Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
import getpass
import re
import shlex

def run(cmd, cwd=None, env=None, log_path=None, display_cmd=None):
    print("$", display_cmd or cmd)
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    log_file = open(log_path, "w", encoding="utf-8") if log_path else None
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=merged_env,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line, end="")
            if log_file:
                log_file.write(line)
                log_file.flush()
    finally:
        if log_file:
            log_file.close()
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {display_cmd or cmd}")

def clean_url(value):
    text = str(value).strip()
    markdown = re.search(r"\((https?://[^)]+)\)", text)
    if markdown:
        return markdown.group(1)
    plain = re.search(r"https?://[^\]\s)]+", text)
    return plain.group(0) if plain else text

def github_token():
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    return token or getpass.getpass("GitHub token, leave empty for public repo: ").strip()

repo_url = clean_url(REPO_URL)
token = github_token() if "github.com" in repo_url else ""
clone_url = repo_url
display_url = repo_url
if token and repo_url.startswith("https://github.com/"):
    clone_url = repo_url.replace("https://", f"https://x-access-token:{token}@", 1)
    display_url = repo_url.replace("https://", "https://<TOKEN>@", 1)

if pathlib.Path(CLONE_DIR).exists():
    shutil.rmtree(CLONE_DIR)

clone_cmd = "git clone --branch {} --single-branch {} {}".format(
    shlex.quote(BRANCH), shlex.quote(clone_url), shlex.quote(CLONE_DIR)
)
display_clone_cmd = "git clone --branch {} --single-branch {} {}".format(
    shlex.quote(BRANCH), shlex.quote(display_url), shlex.quote(CLONE_DIR)
)
run(clone_cmd, display_cmd=display_clone_cmd)
run("git rev-parse HEAD", cwd=CLONE_DIR)
run("git status --short --branch", cwd=CLONE_DIR)

In [ ]:
game24_dir = pathlib.Path(CLONE_DIR) / "game24_composition"
modp_dir = pathlib.Path(CLONE_DIR) / "modp_verifier_sandbox"

if RUN_GAME24:
    run("pip install -q -e .", cwd=str(game24_dir))

if RUN_MODP:
    if not modp_dir.exists():
        raise FileNotFoundError("modp_verifier_sandbox is not on the cloned branch. Push it first or set RUN_MODP=False.")
    run("pip install -q -r requirements.txt", cwd=str(modp_dir))

In [ ]:
if RUN_GAME24:
    game24_env = {}
    if MODE == "smoke":
        game24_env.update({
            "N_TRAIN_A": "80",
            "N_EVAL_A": "20",
            "N_TRAIN_B": "80",
            "N_EVAL_B": "20",
            "N_TEST_AB": "10",
            "EPOCHS": "1",
            "BATCH_SIZE": "1",
            "GRAD_ACCUM": "8",
        })
    run("python -m pip uninstall -y torchao", cwd=str(game24_dir))
    run("bash run_first5.sh", cwd=str(game24_dir), env=game24_env, log_path="/content/game24_first5.log")

In [ ]:
if RUN_MODP:
    modp_env = {}
    if MODE == "smoke":
        modp_env.update({
            "P": "17",
            "EPOCHS": "5",
            "D_MODEL": "32",
            "N_LAYERS": "1",
            "N_HEADS": "2",
            "BATCH_SIZE": "64",
        })
    run("bash run_modp_verifier.sh", cwd=str(modp_dir), env=modp_env, log_path="/content/modp_verifier.log")

In [ ]:
run_id = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_{MODE}"
artifact_dir = pathlib.Path(DRIVE_DIR) / run_id
artifact_dir.mkdir(parents=True, exist_ok=True)

def copy_if_exists(src, dst):
    src = pathlib.Path(src)
    dst = pathlib.Path(dst)
    if not src.exists():
        print("missing, skip:", src)
        return
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

copy_if_exists(pathlib.Path(CLONE_DIR) / ".git" / "HEAD", artifact_dir / "git_HEAD")
run("git rev-parse HEAD > /content/git_commit.txt", cwd=CLONE_DIR)
copy_if_exists("/content/git_commit.txt", artifact_dir / "git_commit.txt")

if RUN_GAME24:
    copy_if_exists(game24_dir / "runs", artifact_dir / "game24_composition" / "runs")
    copy_if_exists(game24_dir / "outputs", artifact_dir / "game24_composition" / "outputs")
    copy_if_exists(game24_dir / "data" / "synthetic", artifact_dir / "game24_composition" / "data_synthetic")
    copy_if_exists("/content/game24_first5.log", artifact_dir / "game24_composition" / "run_first5.log")

if RUN_MODP:
    copy_if_exists(modp_dir / "outputs", artifact_dir / "modp_verifier_sandbox" / "outputs")
    copy_if_exists(modp_dir / "checkpoints", artifact_dir / "modp_verifier_sandbox" / "checkpoints")
    copy_if_exists("/content/modp_verifier.log", artifact_dir / "modp_verifier_sandbox" / "run_modp_verifier.log")

archive_base = str(artifact_dir)
archive_path = shutil.make_archive(archive_base, "gztar", root_dir=artifact_dir.parent, base_dir=artifact_dir.name)
print("Saved artifacts to:", artifact_dir)
print("Archive:", archive_path)